# Analisi Diffusione COVID-19

Questo notebook analizza la diffusione del COVID-19 utilizzando il dataset
**Our World in Data (OWID)**.

Fonte ufficiale:
https://github.com/owid/covid-19-data

Il dataset contiene informazioni giornaliere su casi, ospedalizzazioni,
terapia intensiva e vaccinazioni per diversi paesi e aree aggregate.

Obiettivi dell'analisi:
- Verificare dimensioni e metadati del dataset
- Analizzare i casi totali per continente
- Studiare l'andamento del COVID-19 in Italia nel 2022
- Confrontare i pazienti ICU tra Italia, Germania e Francia
- Analizzare le ospedalizzazioni nel 2021
``

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

## Caricamento del dataset

In [ ]:
url = "https://raw.githubusercontent.com/owid/covid-19-data/refs/heads/master/public/data/owid-covid-data.csv"
df = pd.read_csv(url)

In [ ]:
df.shape

## Esplorazione iniziale del dataset (EDA)

Il dataset contiene sia dati **aggregati** (ad esempio "World", "Europe")
sia dati **specifici per nazione".

Le colonne più rilevanti sono:
- `continent`
- `location`
- `date`
- `total_cases`, `new_cases`
- `icu_patients`, `hosp_patients`

Le righe aggregate non hanno valore nella colonna `continent`.

In [ ]:
df.info()
``

## Analisi dei casi per continente

In [ ]:
continent_cases = (
    df[df["continent"].notna()]
    .groupby(["continent", "date"])["total_cases"]
    .sum()
    .groupby("continent")
    .max()
)

continent_cases
``

In [ ]:
world_total = (
    df[df["location"] == "World"]
    .sort_values("date")["total_cases"]
    .iloc[-1]
)

continent_percentage = (continent_cases / world_total) * 100
continent_percentage

In [ ]:
L’Europa e le Americhe rappresentano la quota maggiore dei casi globali,
mentre Africa e Oceania mostrano percentuali sensibilmente più basse.
``

## Analisi dell'Italia nel 2022

In [ ]:
df_ita_2022 = df[
    (df["location"] == "Italy") &
    (df["date"].str.startswith("2022")) &
    (df["new_cases"].notna())
].copy()

df_ita_2022["date"] = pd.to_datetime(df_ita_2022["date"])

In [ ]:
plt.plot(df_ita_2022["date"], df_ita_2022["total_cases"])
plt.title("Italia – Evoluzione dei casi totali (2022)")
plt.xlabel("Data")
plt.ylabel("Casi totali")
plt.tight_layout()
plt.show()
``

Nel corso del 2022 i casi totali in Italia crescono in modo continuo,
con fasi di incremento più rapido associate alle principali ondate.

In [ ]:
plt.bar(df_ita_2022["date"], df_ita_2022["new_cases"])
plt.title("Italia – Nuovi casi giornalieri (2022)")
plt.xlabel("Data")
plt.ylabel("Nuovi casi")
plt.tight_layout()
plt.show()

In [ ]:
Il numero di nuovi casi presenta forti oscillazioni,
con picchi evidenti in specifici periodi dell’anno.

## Confronto pazienti in terapia intensiva (ICU)

Periodo analizzato:
- Maggio 2022 – Aprile 2023
Paesi:
- Italia
- Germania
- Francia

In [ ]:
countries = ["Italy", "Germany", "France"]

df_icu = df[
    (df["location"].isin(countries)) &
    (df["date"] >= "2022-05-01") &
    (df

In [ ]:
sns.boxplot(data=df_icu, x="location", y="icu_patients")
plt.title("Pazienti in terapia intensiva (05/2022 – 04/2023)")
plt.xlabel("Nazione")
plt.ylabel("Pazienti ICU")
plt.tight_layout()
plt.show()
``

La Germania presenta una mediana più elevata di pazienti ICU.
L’Italia mostra una maggiore variabilità,
mentre la Francia ha valori mediamente più contenuti.

## Ospedalizzazioni nel 2021

Paesi analizzati:
- Italia
- Germania
- Francia
- Spagna

In [ ]:
countries = ["Italy", "Germany", "France", "Spain"]

df_hosp_2021 = df[
    (df["location"].isin(countries)) &
    (df["date"].str.startswith("2021"))
]

hosp_sum = df_hosp_2021.groupby("location")["hosp_patients"].sum()
hosp_sum

In [ ]:
hosp_sum.plot(kind="bar")
plt.title("Somma pazienti ospedalizzati – 2021")
plt.xlabel("Nazione")
plt.ylabel("Somma pazienti ospedalizzati")
plt.tight_layout()
plt.show()

La colonna `hosp_patients` presenta numerosi valori nulli.
Tali valori non risultano distribuiti casualmente,
pertanto non è stata applicata alcuna sostituzione (media o mediana),
per evitare distorsioni nei risultati.

## Conclusioni

- Il dataset OWID è molto dettagliato ma richiede attenzione nella distinzione
  tra dati aggregati e dati specifici per nazione.
- Nel 2022 l'Italia ha mostrato una forte stagionalità nei nuovi casi COVID-19.
- Il confronto ICU evidenzia differenze strutturali tra i sistemi sanitari
  di Italia, Germania e Francia.
- I dati di ospedalizzazione presentano criticità legate ai valori mancanti,
  che rendono necessaria un’interpretazione prudente dei risultati.